# 119番通報 緊急度判定プロトコル — 遷移図

`transition_diagram/protocol.yaml` から症候別の遷移図を Graphviz で生成します。

**レイアウトの取り決め**

| 形状 | 意味 |
|---|---|
| ◇ ひし形 | 選択肢を持つ判定ノード（質問） |
| ▱ 平行四辺形 | 情報収集ノード（`metadata_only`、緊急度に影響しない） |
| ⬭ 楕円 | トリアージ確定ノード（葉） |
| ⬜ 帯状 | 外部プロトコル・外部フローへのルーティング先 |
| 付箋 | プロトコルの fallback ルール |

**色（トリアージ）**

- 赤: R1 / R2 / R3（Very urgent）
- 黄: Y1 / Y2（Semi-urgent）
- 緑: G（Low urgency）

## 1. セットアップ

In [ ]:
from pathlib import Path

import yaml
from graphviz import Digraph
from IPython.display import Markdown, display

YAML_PATH = Path("protocol.yaml")
with YAML_PATH.open(encoding="utf-8") as f:
    data = yaml.safe_load(f)

labels = data["labels"]
entry_flow = {n["id"]: n for n in data["entry_flow"]}
common_vitals = {n["id"]: n for n in data["common_vitals"]}
router = data["chief_complaint_router"]
protocols = {p["id"]: p for p in data["protocols"]}

print(f"YAML: {YAML_PATH.resolve()}")
print(f"プロトコル数: {len(protocols)}")
print(f"共通バイタルノード数: {len(common_vitals)}")
print("プロトコル一覧:")
for pid, p in protocols.items():
    print(f"  - {pid}: {p.get('name', '')}")

## 2. 描画ヘルパー

In [ ]:
TRIAGE_FILL = {
    "R1": "#e74c3c", "R2": "#e74c3c", "R3": "#c0392b",
    "Y1": "#f1c40f", "Y2": "#f4d03f",
    "G": "#27ae60",
}
TRIAGE_FONT = {
    "R1": "white", "R2": "white", "R3": "white",
    "Y1": "black", "Y2": "black",
    "G": "white",
}

FONT = "Hiragino Sans"


def _trim(text: str, n: int = 28) -> str:
    text = (text or "").replace("\n", " ").strip()
    return text if len(text) <= n else text[: n - 1] + "…"


def _new_graph(title: str) -> Digraph:
    g = Digraph()
    g.attr(
        rankdir="TB",
        label=title,
        labelloc="t",
        fontname=FONT,
        fontsize="16",
        splines="spline",
        nodesep="0.35",
        ranksep="0.45",
    )
    g.attr("node", fontname=FONT, fontsize="11", margin="0.12,0.06")
    g.attr("edge", fontname=FONT, fontsize="9")
    return g


def _terminal(g: Digraph, parent: str, edge_label: str, triage: str, oral=None):
    tid = f"__term_{parent}_{len(g.body)}"
    text = triage
    if oral:
        text += "\n口頭指導: " + ", ".join(oral)
    g.node(
        tid,
        text,
        shape="ellipse",
        style="filled,bold",
        fillcolor=TRIAGE_FILL.get(triage, "#bdc3c7"),
        fontcolor=TRIAGE_FONT.get(triage, "black"),
    )
    g.edge(parent, tid, label=_trim(edge_label, 16))


def _question_node(g: Digraph, nid: str, question: str, fill="#fdf2e9"):
    g.node(nid, _trim(question, 30), shape="diamond", style="filled", fillcolor=fill)


def _info_node(g: Digraph, nid: str, question: str):
    g.node(
        nid,
        "[情報収集]\n" + _trim(question, 28),
        shape="parallelogram",
        style="filled",
        fillcolor="#d6eaf8",
    )


def _route_node(g: Digraph, tgt_id: str, label: str):
    g.node(tgt_id, label, shape="cds", style="filled,dashed", fillcolor="#fadbd8")


def _add_choices(g: Digraph, parent: str, choices):
    for ch in choices or []:
        etext = ch.get("text") or ch.get("value") or ch.get("code") or ""
        if ch.get("triage"):
            _terminal(g, parent, etext, ch["triage"], ch.get("oral_instruction"))
        if ch.get("next"):
            g.edge(parent, ch["next"], label=_trim(etext, 16))
        if ch.get("route_to_protocol"):
            tgt = f"__proto_{ch['route_to_protocol']}"
            _route_node(g, tgt, f"→ {ch['route_to_protocol']}")
            g.edge(parent, tgt, label=_trim(etext, 16))
        if ch.get("route_to"):
            tgt = f"__route_{ch['route_to']}"
            _route_node(g, tgt, ch["route_to"])
            g.edge(parent, tgt, label=_trim(etext, 16))
        if ch.get("followup_metadata"):
            g.edge(parent, ch["followup_metadata"], label="followup")

## 3. 導入フロー：エントリ → 共通バイタル → 主訴ルーター

In [ ]:
def render_common_flow() -> Digraph:
    g = _new_graph("導入フロー：エントリ → 共通バイタル → 主訴ルーター")

    for nid, node in entry_flow.items():
        q = node.get("question", nid)
        if node.get("metadata_only") or node.get("transition_only"):
            _info_node(g, nid, q)
        else:
            _question_node(g, nid, q)
        if node.get("next"):
            g.edge(nid, node["next"])
        _add_choices(g, nid, node.get("choices"))

    for nid, node in common_vitals.items():
        q = node.get("question", nid)
        _question_node(g, nid, q, fill="#fcf3cf")
        if node.get("next"):
            g.edge(nid, node["next"])
        _add_choices(g, nid, node.get("choices"))

    _route_node(g, "chief_complaint_router", "主訴ルーター\n(chief_complaint_router)")
    return g


display(render_common_flow())

## 4. 主訴ルーター（症候 → プロトコル）

In [ ]:
def render_router() -> Digraph:
    g = _new_graph("主訴ルーター（chief_complaint_router）")
    g.node("__router", "主訴キーワード照合", shape="diamond", style="filled", fillcolor="#fdf2e9")
    for r in router["routes"]:
        pid = r["protocol"]
        kws = "、".join(r.get("match_any", [])[:4])
        if len(r.get("match_any", [])) > 4:
            kws += "…"
        cond = r.get("age_condition")
        edge_label = kws + (f"\n[{cond}]" if cond else "")
        tgt = f"__proto_{pid}"
        _route_node(g, tgt, f"→ {pid}")
        g.edge("__router", tgt, label=_trim(edge_label, 24))
    return g


display(render_router())

## 5. 症候別プロトコル

In [ ]:
def render_protocol(pid: str, include_common_vitals: bool = False) -> Digraph:
    proto = protocols[pid]
    title = f"{proto.get('name', '')}  ({pid})"
    if proto.get("source_pages"):
        title += f"  — pp.{','.join(map(str, proto['source_pages']))}"
    g = _new_graph(title)

    entry = f"__entry_{pid}"
    if proto.get("start_after_common_vitals"):
        g.node(entry, "共通バイタル後", shape="cds", style="filled", fillcolor="#ecf0f1")
    else:
        g.node(entry, "開始", shape="circle", style="filled", fillcolor="#ecf0f1")

    if include_common_vitals and "common_vitals" in (proto.get("inherits") or []):
        for nid, node in common_vitals.items():
            _question_node(g, nid, node.get("question", nid), fill="#fcf3cf")
            if node.get("next"):
                g.edge(nid, node["next"])
            _add_choices(g, nid, node.get("choices"))
        if "route_to_symptom_inquiry" in common_vitals:
            g.edge(entry, "common_breathing")
            g.edge("route_to_symptom_inquiry", proto.get("start_node") or proto.get("nodes", [{}])[0].get("id"), label=f"→ {pid}")

    nodes = proto.get("nodes", [])
    if nodes and not include_common_vitals:
        g.edge(entry, proto.get("start_node") or nodes[0]["id"])

    for node in nodes:
        nid = node["id"]
        q = node.get("question", nid)

        if node.get("triage") and not node.get("choices"):
            t = node["triage"]
            g.node(
                nid,
                _trim(q, 30) + f"\n→ {t}",
                shape="ellipse",
                style="filled,bold",
                fillcolor=TRIAGE_FILL.get(t, "#bdc3c7"),
                fontcolor=TRIAGE_FONT.get(t, "black"),
            )
            continue

        if node.get("metadata_only") or node.get("transition_only"):
            _info_node(g, nid, q)
        else:
            _question_node(g, nid, q)
        if node.get("next"):
            g.edge(nid, node["next"])
        _add_choices(g, nid, node.get("choices"))

    followups = proto.get("terminal_followup_chain") or []
    for src, dst in zip(followups, followups[1:]):
        g.edge(src, dst, label="followup")

    fb = proto.get("fallback")
    if fb:
        parts = []
        if "if_any_unknown" in fb:
            parts.append(f"不明あり → {fb['if_any_unknown']}")
        if "if_all_symptom_questions_negative" in fb:
            parts.append(f"全て陰性 → {fb['if_all_symptom_questions_negative']}")
        g.node(f"__fb_{pid}", "fallback\n" + "\n".join(parts), shape="note", style="filled", fillcolor="#fdebd0")

    return g

### 5.1 個別表示

In [ ]:
protocol_id = "dyspnea"

display(Markdown(f"#### {protocols[protocol_id].get('name', protocol_id)} (`{protocol_id}`)"))
display(render_protocol(protocol_id))

### 5.2 共通バイタル込みで表示

In [ ]:
protocol_id = "dyspnea"

display(Markdown(f"#### 共通バイタル込み: {protocols[protocol_id].get('name', protocol_id)} (`{protocol_id}`)"))
display(render_protocol(protocol_id, include_common_vitals=True))

### 5.3 全プロトコル一括表示

In [ ]:
for pid, proto in protocols.items():
    display(Markdown(f"#### {proto.get('name', pid)} (`{pid}`)"))
    display(render_protocol(pid))

## 6. ファイル保存（任意）

SVG と PDF を `graphviz_diagrams/` に書き出します。

In [ ]:
OUT_DIR = Path("graphviz_diagrams")
OUT_DIR.mkdir(exist_ok=True)

render_common_flow().render(OUT_DIR / "00_common_flow", format="svg", cleanup=True)
render_router().render(OUT_DIR / "01_router", format="svg", cleanup=True)

for i, pid in enumerate(protocols, start=2):
    render_protocol(pid).render(OUT_DIR / f"{i:02d}_{pid}", format="svg", cleanup=True)

print(f"saved to {OUT_DIR.resolve()}")